# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL, 브랜치 이름, GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    branch_name = input("Git 브랜치 이름 (예: main / 기본 브랜치면 Enter): ").strip()
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        clone_cmd = ["git", "clone"]
        if branch_name:
            clone_cmd.extend(["-b", branch_name])
        clone_cmd.extend([clone_url, str(repo_dir)])
        subprocess.run(clone_cmd, check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")
        subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
        if not branch_name:
            branch_name = subprocess.run(
                ["git", "branch", "--show-current"],
                cwd=repo_dir,
                text=True,
                capture_output=True,
                check=True,
            ).stdout.strip() or "main"
        subprocess.run(["git", "checkout", branch_name], cwd=repo_dir, check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", branch_name], cwd=repo_dir, check=True)

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))

torch_shadow_candidates = [
    repo_dir / "torch.py",
    repo_dir / "torch",
    repo_dir / "src" / "torch.py",
    repo_dir / "src" / "torch",
]
torch_shadows = [str(path) for path in torch_shadow_candidates if path.exists()]
if torch_shadows:
    raise RuntimeError(f"PyTorch import를 가리는 파일/폴더가 있습니다: {torch_shadows}")

try:
    import torch
    import torch.nn as nn
except ImportError:
    # Colab에서 torch import가 한 번 실패하면 sys.modules에 반쯤 로드된 torch가 남을 수 있습니다.
    for module_name in list(sys.modules):
        if module_name == "torch" or module_name.startswith("torch."):
            del sys.modules[module_name]
    import torch
    import torch.nn as nn

current_branch = subprocess.run(["git", "branch", "--show-current"], cwd=repo_dir, text=True, capture_output=True).stdout.strip()
print(f"Repo: {repo_dir}")
print(f"Branch: {current_branch or '(detached HEAD)'}")
print("PyTorch:", torch.__version__, Path(torch.__file__).resolve())

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")

## 10. BPE vocab 생성/로드

10번 셀은 `nsmc_bpe_vocab_3000.json`을 먼저 찾아서 그대로 로드합니다. 찾는 순서는 Google Drive, repo의 `data/`, 현재 폴더, Colab `/content`입니다.

파일이 없으면 `corpus[:1_500_000]`과 `vocab_size=3000`으로 새로 학습해 repo `data/`와 가능한 경우 Google Drive에 저장합니다. 따라서 11번 셀은 항상 이 셀에서 준비된 `tokenizer`를 사용합니다.


In [ ]:
# 10. BPE vocab 생성/로드
import shutil
import sys
import time
from pathlib import Path

from bpe import BPETokenizer


# 과제 Basic 기준: corpus[:1_500_000], vocab_size=3000
VOCAB_SIZE = 3000
BPE_TRAIN_CHARS = 1_500_000
VOCAB_FILENAME = f"nsmc_bpe_vocab_{VOCAB_SIZE}.json"
VOCAB_PATH = Path(repo_dir) / "data" / VOCAB_FILENAME

# Colab에서는 로컬에서 만든 vocab을 올려서 쓰는 흐름이 기본입니다.
# 로컬에서는 파일이 없으면 학습하고, Colab에서는 파일이 없으면 업로드 안내를 냅니다.
IN_COLAB = "google.colab" in sys.modules
TRAIN_BPE_IF_MISSING = True
FORCE_RETRAIN_BPE = False
DRIVE_VOCAB_PATH = Path("/content/drive/MyDrive/W14-gpt-lab/data") / VOCAB_FILENAME if IN_COLAB else None
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    print(f"Drive vocab path: {DRIVE_VOCAB_PATH}")


def find_vocab_file() -> Path | None:
    """Drive, repo data/, Colab /content에서 vocab 후보를 찾습니다."""
    candidates = [
        DRIVE_VOCAB_PATH,
        VOCAB_PATH,
        Path.cwd() / VOCAB_FILENAME,
        Path("/content") / VOCAB_FILENAME,
    ]
    for candidate in candidates:
        if candidate is not None:
            candidate = Path(candidate)
            if candidate.exists():
                return candidate
    return None


if "corpus" not in globals() or not corpus:
    LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
    if not LM_TRAIN_PATH.exists():
        raise FileNotFoundError("먼저 2번 NSMC 데이터 준비 셀을 실행하세요.")
    corpus = LM_TRAIN_PATH.read_text(encoding="utf-8")

start_time = time.time()
tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)
existing_vocab = find_vocab_file()

if existing_vocab is not None and not FORCE_RETRAIN_BPE:
    tokenizer.load(existing_vocab)
    if len(tokenizer.id_to_token) != VOCAB_SIZE:
        message = f"vocab 크기 불일치: {existing_vocab} has {len(tokenizer.id_to_token)}, expected {VOCAB_SIZE}"
        if TRAIN_BPE_IF_MISSING:
            print(message)
            print("로컬이므로 Basic vocab을 새로 학습합니다.")
            existing_vocab = None
        else:
            raise ValueError(message + "입니다. Basic용 vocab 파일을 업로드하세요.")
    else:
        # Colab에 /content/nsmc_bpe_vocab_3000.json처럼 올린 경우 repo data/로 복사해
        # 11번 셀도 같은 경로에서 안정적으로 찾게 합니다.
        VOCAB_PATH.parent.mkdir(parents=True, exist_ok=True)
        if existing_vocab.resolve() != VOCAB_PATH.resolve():
            shutil.copy2(existing_vocab, VOCAB_PATH)
            print(f"업로드 vocab 복사: {existing_vocab} -> {VOCAB_PATH}")
        print(f"기존 vocab 로드: {existing_vocab}")

if existing_vocab is None and (TRAIN_BPE_IF_MISSING or FORCE_RETRAIN_BPE):
    train_text = corpus[:BPE_TRAIN_CHARS]
    print(f"BPE 학습 시작: chars={len(train_text):,}, vocab_size={VOCAB_SIZE}")
    tokenizer.train(train_text)
    tokenizer.save(VOCAB_PATH)
    print(f"새 vocab 저장: {VOCAB_PATH}")
    if DRIVE_VOCAB_PATH is not None:
        DRIVE_VOCAB_PATH.parent.mkdir(parents=True, exist_ok=True)
        tokenizer.save(DRIVE_VOCAB_PATH)
        print(f"Drive vocab 저장: {DRIVE_VOCAB_PATH}")
elif existing_vocab is None:
    raise FileNotFoundError(
        "Colab에서는 로컬에서 만든 vocab 파일을 먼저 업로드하세요. "
        f"권장 파일명: {VOCAB_FILENAME}"
    )

print("vocab_size 설정:", tokenizer.vocab_size)
print("실제 token 수:", len(tokenizer.id_to_token))
print("merge 수:", len(tokenizer.merges))
print(f"소요 시간: {time.time() - start_time:.1f}초")

sample = "이 영화는 좋았다"
sample_ids = tokenizer.encode(sample, add_bos_eos=True)
print("sample ids:", sample_ids[:20])
print("decode:", tokenizer.decode(sample_ids))

## 11. Basic 사전 학습 실행

10번에서 로드하거나 생성한 BPE vocab을 사용해 참조 브랜치와 같은 Basic 설정으로 mini GPT를 사전 학습합니다. 기본값은 `vocab_size=3000`, `context_length=128`, `emb_dim=128`, `n_layers=2`, `batch_size=8`, `num_epochs=3`입니다.

Colab/Drive가 있으면 token id cache, checkpoint, loss curve, summary JSON을 `MyDrive/W14-gpt-lab/` 아래에 저장하고, 로컬에서는 `cache/`, `checkpoints/`, `runs/`를 사용합니다.


In [ ]:
# 11. Basic 사전 학습 실행
import json
import shutil
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from dataset import create_dataloader
from model import GPTModel
from train import train_model, calc_loss_loader

cell_start_time = time.perf_counter()

# 10번 셀에서 vocabulary를 load/create한 tokenizer를 그대로 사용합니다.
# tokenizer가 없다면 먼저 BPE vocabulary load 셀을 실행하세요.
assert "tokenizer" in globals(), "먼저 BPE vocabulary load 셀을 실행해 tokenizer를 준비하세요."
assert len(tokenizer.id_to_token) == 3000, f"expected vocab size 3000, got {len(tokenizer.id_to_token)}"
assert "corpus" in globals() and len(corpus) > 0, "먼저 NSMC LM train corpus를 로드하세요."

# 전체 corpus로 최종 사전 학습을 시작합니다.
# 빠른 탐색이 필요하면 TRAIN_CHAR_LIMIT를 300_000 또는 500_000으로 낮춰서 먼저 비교하세요.
TRAIN_CHAR_LIMIT = None
VAL_CHAR_LIMIT = None

train_text = corpus if TRAIN_CHAR_LIMIT is None else corpus[:TRAIN_CHAR_LIMIT]
if "val_corpus" in globals() and val_corpus:
    val_text = val_corpus if VAL_CHAR_LIMIT is None else val_corpus[:VAL_CHAR_LIMIT]
else:
    split_idx = int(len(train_text) * 0.9)
    val_text = train_text[split_idx:]
    train_text = train_text[:split_idx]

context_length = 128
batch_size = 8
learning_rate = 3e-4
num_epochs = 3
eval_freq = 100
eval_iter = 20
ckpt_freq = 1
encode_each_line_with_bos_eos = True

config = {
    "vocab_size": len(tokenizer.id_to_token),
    "context_length": context_length,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

def encode_lm_text(text: str, add_bos_eos_per_line: bool) -> list[int]:
    if not add_bos_eos_per_line:
        return tokenizer.encode(text)

    token_ids = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            token_ids.extend(tokenizer.encode(line, add_bos_eos=True))
    return token_ids


cache_root = Path("/content/drive/MyDrive/W14-gpt-lab/cache")
if not cache_root.exists():
    cache_root = Path("cache")
cache_root.mkdir(parents=True, exist_ok=True)

bos_tag = "bos" if encode_each_line_with_bos_eos else "nobos"
train_limit_tag = "full" if TRAIN_CHAR_LIMIT is None else str(TRAIN_CHAR_LIMIT)
val_limit_tag = "full" if VAL_CHAR_LIMIT is None else str(VAL_CHAR_LIMIT)
cache_name = f"bpe{len(tokenizer.id_to_token)}_{bos_tag}_train{train_limit_tag}_val{val_limit_tag}.pt"
cache_path = cache_root / cache_name

encode_start_time = time.perf_counter()
if cache_path.exists():
    cached = torch.load(cache_path, map_location="cpu")
    train_ids = cached["train_ids"]
    val_ids = cached["val_ids"]
    print("loaded token id cache:", cache_path)
else:
    train_ids = encode_lm_text(train_text, encode_each_line_with_bos_eos)
    val_ids = encode_lm_text(val_text, encode_each_line_with_bos_eos)
    torch.save({"train_ids": train_ids, "val_ids": val_ids}, cache_path)
    print("saved token id cache:", cache_path)
encode_elapsed = time.perf_counter() - encode_start_time

train_loader = create_dataloader(
    train_ids,
    context_length=context_length,
    batch_size=batch_size,
    shuffle=True,
)
val_loader = create_dataloader(
    val_ids,
    context_length=context_length,
    batch_size=batch_size,
    shuffle=False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("train chars:", len(train_text), "val chars:", len(val_text))
print("train tokens:", len(train_ids), "val tokens:", len(val_ids))
print("train batches:", len(train_loader), "val batches:", len(val_loader))
print("config:", config)
print("batch_size:", batch_size, "learning_rate:", learning_rate, "num_epochs:", num_epochs)
print("encode_each_line_with_bos_eos:", encode_each_line_with_bos_eos)
print("encode_time_sec:", round(encode_elapsed, 2))
print("encode_time_min:", round(encode_elapsed / 60, 2))

model = GPTModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

start_time = time.perf_counter()
train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=eval_freq,
    eval_iter=eval_iter,
    start_context="이 영화는",
    tokenizer=tokenizer,
    ckpt_freq=ckpt_freq,
)
elapsed = time.perf_counter() - start_time

print("training_loop_time_sec:", round(elapsed, 2))
print("training_loop_time_min:", round(elapsed / 60, 2))
final_full_val_loss = calc_loss_loader(val_loader, model, device, num_batches=None)
total_elapsed = time.perf_counter() - cell_start_time

print("best_train_loss:", min(train_losses) if train_losses else None)
print("best_val_loss_estimate:", min(val_losses) if val_losses else None)
print("final_train_loss_estimate:", train_losses[-1] if train_losses else None)
print("final_val_loss_estimate:", val_losses[-1] if val_losses else None)
print("final_full_val_loss:", final_full_val_loss)
print("total_cell_time_sec:", round(total_elapsed, 2))
print("total_cell_time_min:", round(total_elapsed / 60, 2))

eval_steps = [(i + 1) * eval_freq for i in range(len(train_losses))]
eval_epochs = [((step - 1) // len(train_loader)) + 1 for step in eval_steps]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(eval_steps, train_losses, marker="o", label="Train")
ax.plot(eval_steps, val_losses, marker="o", label="Val")
ax.set_xlabel("Global step / Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training / Validation Loss")
ax.legend()
ax.grid(alpha=0.25)

if eval_steps:
    tick_count = min(8, len(eval_steps))
    tick_indices = sorted(set(round(i * (len(eval_steps) - 1) / max(1, tick_count - 1)) for i in range(tick_count)))
    tick_values = [eval_steps[i] for i in tick_indices]
    tick_labels = [f"{eval_steps[i]}\nE{eval_epochs[i]}" for i in tick_indices]
    ax.set_xticks(tick_values)
    ax.set_xticklabels(tick_labels)

fig.tight_layout()
local_plot_path = Path("loss_curve.png")
fig.savefig(local_plot_path, dpi=150, bbox_inches="tight")
plt.show()

# Colab 런타임 저장소의 checkpoints/는 사라질 수 있으므로 Drive에도 복사합니다.
drive_root = Path("/content/drive/MyDrive/W14-gpt-lab")
if drive_root.exists():
    bos_tag = "bos" if encode_each_line_with_bos_eos else "nobos"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = (
        f"{timestamp}_pretrain"
        f"_ctx{context_length}"
        f"_{bos_tag}"
        f"_emb{config['emb_dim']}"
        f"_heads{config['n_heads']}"
        f"_layers{config['n_layers']}"
        f"_drop{config['drop_rate']}"
        f"_bs{batch_size}"
        f"_lr{learning_rate:g}"
        f"_ep{num_epochs}"
    )
    drive_run_dir = drive_root / "runs" / run_name

    suffix = 1
    while drive_run_dir.exists():
        drive_run_dir = drive_root / "runs" / f"{run_name}_repeat{suffix}"
        suffix += 1

    drive_ckpt_dir = drive_run_dir / "checkpoints"
    drive_ckpt_dir.mkdir(parents=True, exist_ok=False)

    drive_plot_path = drive_run_dir / "loss_curve.png"
    if local_plot_path.exists():
        shutil.copy2(local_plot_path, drive_plot_path)

    for epoch in range(1, num_epochs + 1):
        ckpt_path = Path("checkpoints") / f"ckpt_epoch_{epoch}.pt"
        if ckpt_path.exists():
            shutil.copy2(ckpt_path, drive_ckpt_dir / ckpt_path.name)

    summary = {
        "run_name": run_name,
        "run_dir": str(drive_run_dir),
        "train_chars": len(train_text),
        "val_chars": len(val_text),
        "train_tokens": len(train_ids),
        "val_tokens": len(val_ids),
        "token_cache_path": str(cache_path),
        "train_batches": len(train_loader),
        "val_batches": len(val_loader),
        "config": config,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "num_epochs": num_epochs,
        "eval_freq": eval_freq,
        "eval_iter": eval_iter,
        "ckpt_freq": ckpt_freq,
        "encode_each_line_with_bos_eos": encode_each_line_with_bos_eos,
        "encode_time_sec": round(encode_elapsed, 2),
        "training_loop_time_sec": round(elapsed, 2),
        "total_cell_time_sec": round(total_elapsed, 2),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "eval_steps": eval_steps,
        "eval_epochs": eval_epochs,
        "loss_curve_path": str(drive_plot_path),
        "best_train_loss": min(train_losses) if train_losses else None,
        "best_val_loss_estimate": min(val_losses) if val_losses else None,
        "final_train_loss_estimate": train_losses[-1] if train_losses else None,
        "final_val_loss_estimate": val_losses[-1] if val_losses else None,
        "final_full_val_loss": final_full_val_loss,
    }

    with open(drive_run_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("saved run summary:", drive_run_dir / "summary.json")
    print("saved loss curve:", drive_plot_path)
    print("saved checkpoints:", drive_ckpt_dir)
else:
    print("Drive path not found. Checkpoints remain in local ./checkpoints")


## 12. 감성 분류 미세 조정 실행

11번에서 사전 학습한 GPT backbone에 `finetune.py`의 `GPTForSequenceClassification` head를 붙여 NSMC 긍정/부정 분류를 학습합니다.

결과는 epoch별 train/validation loss·accuracy, 최종 test loss·accuracy, confusion matrix, 오분류 예시로 한눈에 확인합니다.


In [ ]:
# 12. 감성 분류 미세 조정 실행
import copy
import json
import random
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from bpe import BPETokenizer
from model import GPTModel
from finetune import (
    ReviewSentimentDataset,
    GPTForSequenceClassification,
    apply_lora_to_gpt,
    count_lora_parameters,
    evaluate_sentiment,
    train_epoch_sentiment,
)

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None

# -----------------------------------------------------------------------------
# 1. 실행 설정
# -----------------------------------------------------------------------------
SENTIMENT_SEED = 123
SENTIMENT_MAX_LENGTH = 128
SENTIMENT_BATCH_SIZE = 32
SENTIMENT_NUM_EPOCHS = 3
SENTIMENT_FREEZE_BACKBONE = True
SENTIMENT_UNFREEZE_LAST_N_BLOCKS = 0
SENTIMENT_TRAIN_FINAL_LAYERNORM = False
SENTIMENT_USE_LORA = True
SENTIMENT_LORA_RANK = 8
SENTIMENT_LORA_ALPHA = 16.0
SENTIMENT_LORA_DROPOUT = 0.05
SENTIMENT_LORA_TARGET_MODULES = ("W_q", "W_v")
SENTIMENT_BACKBONE_LR = 1e-5
SENTIMENT_LORA_LR = 3e-4
SENTIMENT_CLASSIFIER_LR = 1e-4
SENTIMENT_WEIGHT_DECAY = 0.01

# 빠른 smoke run이 필요하면 숫자를 넣으세요. 최종 결과는 None으로 전체 데이터를 사용합니다.
SENTIMENT_TRAIN_LIMIT = None
SENTIMENT_VAL_LIMIT = None
SENTIMENT_TEST_LIMIT = None
SENTIMENT_USE_TOKEN_CACHE = True

random.seed(SENTIMENT_SEED)
torch.manual_seed(SENTIMENT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SENTIMENT_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# -----------------------------------------------------------------------------
# 2. tokenizer와 pretrained GPT backbone 준비
# -----------------------------------------------------------------------------
def find_vocab_file_for_sentiment() -> Path:
    candidates = []
    if "VOCAB_PATH" in globals():
        candidates.append(Path(VOCAB_PATH))
    if "DRIVE_VOCAB_PATH" in globals() and DRIVE_VOCAB_PATH is not None:
        candidates.append(Path(DRIVE_VOCAB_PATH))
    candidates.extend([
        Path(repo_dir) / "data" / "nsmc_bpe_vocab_3000.json",
        Path("data") / "nsmc_bpe_vocab_3000.json",
        Path("/content/drive/MyDrive/W14-gpt-lab/data/nsmc_bpe_vocab_3000.json"),
        Path("/content/nsmc_bpe_vocab_3000.json"),
    ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("BPE vocab을 찾지 못했습니다. 먼저 10번 셀을 실행하세요.")

if "tokenizer" not in globals() or len(getattr(tokenizer, "id_to_token", {})) == 0:
    vocab_path = find_vocab_file_for_sentiment()
    tokenizer = BPETokenizer(vocab_size=3000)
    tokenizer.load(vocab_path)
    print("tokenizer loaded:", vocab_path)
else:
    print("tokenizer ready:", len(tokenizer.id_to_token), "tokens")

if "config" not in globals():
    config = {
        "vocab_size": len(tokenizer.id_to_token),
        "context_length": SENTIMENT_MAX_LENGTH,
        "emb_dim": 128,
        "n_heads": 4,
        "n_layers": 2,
        "drop_rate": 0.1,
        "qkv_bias": False,
    }


# 11번 사전 학습 결과 중 가장 좋은 checkpoint를 우선 사용합니다.
# Colab에서는 11번 셀이 저장한 Drive runs/ 아래에서 ckpt_epoch_30.pt를 찾습니다.
PREFERRED_PRETRAIN_CKPT_NAME = "ckpt_epoch_30.pt"
PREFERRED_PRETRAIN_CKPT_PATHS = [
    Path("/content/drive/MyDrive/W14-gpt-lab/runs") / PREFERRED_PRETRAIN_CKPT_NAME,
    Path("/content/drive/MyDrive/W14-gpt-lab/checkpoints") / PREFERRED_PRETRAIN_CKPT_NAME,
    Path("/content/drive/MyDrive/W14-gpt-lab") / PREFERRED_PRETRAIN_CKPT_NAME,
    Path("/content/drive/MyDrive") / PREFERRED_PRETRAIN_CKPT_NAME,
    Path("/content") / PREFERRED_PRETRAIN_CKPT_NAME,
    Path("checkpoints") / PREFERRED_PRETRAIN_CKPT_NAME,
    Path(repo_dir) / "checkpoints" / PREFERRED_PRETRAIN_CKPT_NAME,
    Path("/Users/kyumin0412/Works/jungle/ai_study/notes") / PREFERRED_PRETRAIN_CKPT_NAME,
]


def sorted_existing_matches(pattern_roots: list[Path], filename: str) -> list[Path]:
    matches = []
    for root in pattern_roots:
        if root.exists():
            matches.extend(root.glob(f"**/{filename}"))
    return sorted(set(matches), key=lambda path: path.stat().st_mtime, reverse=True)


def find_preferred_pretrain_checkpoint() -> Path | None:
    dynamic_candidates = []
    for variable_name in ["drive_ckpt_dir", "drive_run_dir", "OUTPUT_DIR", "run_dir"]:
        value = globals().get(variable_name)
        if value is not None:
            value = Path(value)
            dynamic_candidates.extend([
                value / PREFERRED_PRETRAIN_CKPT_NAME,
                value / "checkpoints" / PREFERRED_PRETRAIN_CKPT_NAME,
            ])

    for candidate in dynamic_candidates + PREFERRED_PRETRAIN_CKPT_PATHS:
        if candidate.exists():
            return candidate

    search_roots = [
        Path("/content/drive/MyDrive/W14-gpt-lab"),
        Path("/content/drive/MyDrive"),
        Path("/content"),
        Path("."),
    ]
    matches = sorted_existing_matches(search_roots, PREFERRED_PRETRAIN_CKPT_NAME)
    if matches:
        print("preferred checkpoint candidates:")
        for match in matches[:5]:
            print("-", match)
        return matches[0]

    return None


def find_latest_pretrain_checkpoint() -> Path | None:
    candidates = []
    candidates.extend(Path("checkpoints").glob("ckpt_epoch_*.pt"))
    drive_runs = Path("/content/drive/MyDrive/W14-gpt-lab/runs")
    if drive_runs.exists():
        candidates.extend(drive_runs.glob("**/checkpoints/ckpt_epoch_*.pt"))
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)

def unwrap_model_state_dict(checkpoint: dict) -> dict:
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    return checkpoint


def infer_gpt_config_from_state_dict(state_dict: dict, fallback_config: dict) -> dict:
    token_weight = state_dict.get("embedding.token_embedding.weight")
    if token_weight is None:
        token_weight = state_dict.get("embedding.token_embedding_layer.weight")
    pos_weight = state_dict.get("embedding.position_embedding.weight")
    if pos_weight is None:
        pos_weight = state_dict.get("embedding.pos_embedding_layer.weight")
    lm_head_weight = state_dict.get("lm_head.weight")

    if token_weight is None and lm_head_weight is None:
        raise KeyError("checkpoint에서 token embedding/lm_head weight를 찾지 못했습니다.")
    if pos_weight is None:
        raise KeyError("checkpoint에서 position embedding weight를 찾지 못했습니다.")

    vocab_size, emb_dim = (token_weight.shape if token_weight is not None else lm_head_weight.shape)
    context_length = pos_weight.shape[0]
    layer_ids = {
        int(key.split(".")[1])
        for key in state_dict
        if key.startswith("trf_blocks.") and key.split(".")[1].isdigit()
    }

    n_heads = int(fallback_config.get("n_heads", 4))
    if emb_dim % n_heads != 0:
        n_heads = next(candidate for candidate in [8, 6, 4, 3, 2, 1] if emb_dim % candidate == 0)

    qkv_bias = any(
        key.endswith(("attention.W_q.bias", "att.W_query.bias"))
        for key in state_dict
    )

    return {
        "vocab_size": int(vocab_size),
        "context_length": int(context_length),
        "emb_dim": int(emb_dim),
        "n_heads": n_heads,
        "n_layers": len(layer_ids) if layer_ids else int(fallback_config.get("n_layers", 2)),
        "drop_rate": float(fallback_config.get("drop_rate", 0.1)),
        "qkv_bias": bool(qkv_bias),
    }


def convert_checkpoint_keys_for_current_model(state_dict: dict) -> dict:
    converted = {}
    for key, value in state_dict.items():
        new_key = key
        new_key = new_key.replace("embedding.token_embedding_layer.", "embedding.token_embedding.")
        new_key = new_key.replace("embedding.pos_embedding_layer.", "embedding.position_embedding.")
        new_key = new_key.replace(".att.W_query.", ".attention.W_q.")
        new_key = new_key.replace(".att.W_key.", ".attention.W_k.")
        new_key = new_key.replace(".att.W_value.", ".attention.W_v.")
        new_key = new_key.replace(".att.out_proj.", ".attention.out_proj.")
        new_key = new_key.replace(".norm1.", ".layernorm1.")
        new_key = new_key.replace(".norm2.", ".layernorm2.")
        new_key = new_key.replace(".ffn.layers.0.", ".ffn.input_layer.")
        new_key = new_key.replace(".ffn.layers.2.", ".ffn.output_layer.")
        converted[new_key] = value

        if ".ffn.input_layer." in new_key:
            converted[new_key.replace(".ffn.input_layer.", ".ffn.net.0.")] = value
        elif ".ffn.output_layer." in new_key:
            converted[new_key.replace(".ffn.output_layer.", ".ffn.net.2.")] = value

    return converted


def load_pretrained_backbone_from_checkpoint(ckpt_path: Path, fallback_config: dict) -> tuple[GPTModel, dict]:
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    state_dict = unwrap_model_state_dict(checkpoint)
    ckpt_config = checkpoint.get("config") if isinstance(checkpoint, dict) else None
    if ckpt_config is None:
        ckpt_config = infer_gpt_config_from_state_dict(state_dict, fallback_config)

    ckpt_config = dict(ckpt_config)
    ckpt_config["vocab_size"] = len(tokenizer.id_to_token)

    backbone = GPTModel(ckpt_config)
    converted_state_dict = convert_checkpoint_keys_for_current_model(state_dict)
    backbone.load_state_dict(converted_state_dict)

    print("pretrained config:", ckpt_config)
    return backbone, ckpt_config

ckpt_path = find_preferred_pretrain_checkpoint()
if ckpt_path is not None:
    backbone, config = load_pretrained_backbone_from_checkpoint(ckpt_path, config)
    print("pretrained backbone loaded:", ckpt_path)
elif "model" in globals() and isinstance(model, GPTModel):
    backbone = model
    config = dict(model.config)
    print("preferred ckpt_epoch_30.pt not found; using model from 11번 셀 memory")
else:
    ckpt_path = find_latest_pretrain_checkpoint()
    if ckpt_path is None:
        checked_roots = [
            "/content/drive/MyDrive/W14-gpt-lab",
            "/content/drive/MyDrive",
            "/content",
            str(Path.cwd()),
        ]
        raise FileNotFoundError(
            "사전 학습 checkpoint를 찾지 못했습니다. "
            f"{PREFERRED_PRETRAIN_CKPT_NAME}을 11번 셀의 Drive runs 폴더나 "
            "/content/drive/MyDrive/W14-gpt-lab/ 아래에 올려주세요. "
            f"확인한 위치: {checked_roots}"
        )
    backbone, config = load_pretrained_backbone_from_checkpoint(ckpt_path, config)
    print("preferred ckpt_epoch_30.pt not found; latest pretrained backbone loaded:", ckpt_path)

SENTIMENT_EFFECTIVE_MAX_LENGTH = min(SENTIMENT_MAX_LENGTH, config["context_length"])
if SENTIMENT_EFFECTIVE_MAX_LENGTH != SENTIMENT_MAX_LENGTH:
    print(
        "sentiment max_length adjusted:",
        SENTIMENT_MAX_LENGTH,
        "->",
        SENTIMENT_EFFECTIVE_MAX_LENGTH,
        "(checkpoint context_length)",
    )

sentiment_model = GPTForSequenceClassification(backbone, num_labels=2, drop_rate=0.1).to(device)
if SENTIMENT_USE_LORA:
    lora_info = apply_lora_to_gpt(
        sentiment_model.gpt,
        target_modules=SENTIMENT_LORA_TARGET_MODULES,
        rank=SENTIMENT_LORA_RANK,
        alpha=SENTIMENT_LORA_ALPHA,
        dropout=SENTIMENT_LORA_DROPOUT,
    )
else:
    lora_info = {
        "num_adapters": 0,
        "rank": 0,
        "alpha": 0.0,
        "target_modules": [],
        "parameters": 0,
    }
print("LoRA:", lora_info)

def count_parameters(module, trainable_only: bool = False) -> int:
    params = module.parameters()
    if trainable_only:
        params = (param for param in params if param.requires_grad)
    return sum(param.numel() for param in params)


def configure_sentiment_trainable_parameters(model: GPTForSequenceClassification) -> dict:
    for param in model.gpt.parameters():
        param.requires_grad = not SENTIMENT_FREEZE_BACKBONE

    if SENTIMENT_FREEZE_BACKBONE and SENTIMENT_UNFREEZE_LAST_N_BLOCKS > 0:
        blocks = list(model.gpt.blocks)
        for block in blocks[-SENTIMENT_UNFREEZE_LAST_N_BLOCKS:]:
            for param in block.parameters():
                param.requires_grad = True

    if SENTIMENT_FREEZE_BACKBONE and SENTIMENT_TRAIN_FINAL_LAYERNORM:
        for param in model.gpt.finalnorm.parameters():
            param.requires_grad = True

    if SENTIMENT_USE_LORA:
        for name, param in model.gpt.named_parameters():
            if ".lora_A." in name or ".lora_B." in name:
                param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

    gpt_trainable = count_parameters(model.gpt, trainable_only=True)
    lora_trainable = count_lora_parameters(model.gpt, trainable_only=True)
    base_gpt_trainable = gpt_trainable - lora_trainable
    classifier_trainable = count_parameters(model.classifier, trainable_only=True)
    total_trainable = count_parameters(model, trainable_only=True)
    return {
        "gpt_trainable": gpt_trainable,
        "base_gpt_trainable": base_gpt_trainable,
        "lora_trainable": lora_trainable,
        "classifier_trainable": classifier_trainable,
        "total_trainable": total_trainable,
    }


trainable_counts = configure_sentiment_trainable_parameters(sentiment_model)
print("trainable parameters:", trainable_counts)
if SENTIMENT_FREEZE_BACKBONE and SENTIMENT_USE_LORA and trainable_counts["base_gpt_trainable"] == 0:
    print("GPT backbone frozen: LoRA adapters + classifier head만 학습합니다.")
elif SENTIMENT_FREEZE_BACKBONE and trainable_counts["gpt_trainable"] == 0:
    print("GPT backbone frozen: classifier head만 학습합니다.")
elif SENTIMENT_FREEZE_BACKBONE:
    print(
        "GPT backbone partially frozen:",
        f"last {SENTIMENT_UNFREEZE_LAST_N_BLOCKS} block(s)",
        "and final layernorm" if SENTIMENT_TRAIN_FINAL_LAYERNORM else "",
    )
else:
    print("GPT backbone trainable: full fine-tuning mode입니다.")

# -----------------------------------------------------------------------------
# 3. NSMC sentiment 데이터 로드
# -----------------------------------------------------------------------------
def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def get_sentiment_token_cache_root() -> Path:
    drive_project_root = Path("/content/drive/MyDrive/W14-gpt-lab")
    cache_root = drive_project_root / "cache" if drive_project_root.exists() else Path("cache")
    cache_root.mkdir(parents=True, exist_ok=True)
    return cache_root


def torch_load_cache(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def pad_or_truncate_token_ids(token_ids: list[int], max_length: int, pad_id: int) -> list[int]:
    token_ids = token_ids[:max_length]
    if len(token_ids) < max_length:
        token_ids = token_ids + [pad_id] * (max_length - len(token_ids))
    return token_ids


def load_or_build_tokenized_sentiment_split(split_name: str, rows: list[dict]) -> list[dict]:
    if not SENTIMENT_USE_TOKEN_CACHE:
        return rows

    cache_root = get_sentiment_token_cache_root()
    pad_id = tokenizer.get_pad_id()
    cache_metadata = {
        "split": split_name,
        "vocab_size": len(tokenizer.id_to_token),
        "max_length": SENTIMENT_EFFECTIVE_MAX_LENGTH,
        "num_rows": len(rows),
        "pad_id": pad_id,
    }
    cache_path = cache_root / (
        f"nsmc_sentiment_tokens_{split_name}"
        f"_v{cache_metadata['vocab_size']}"
        f"_len{cache_metadata['max_length']}"
        f"_n{cache_metadata['num_rows']}.pt"
    )

    if cache_path.exists():
        payload = torch_load_cache(cache_path)
        if payload.get("metadata") == cache_metadata:
            print(f"token cache loaded [{split_name}]:", cache_path)
            return payload["data"]
        print(f"token cache metadata mismatch [{split_name}], rebuilding:", cache_path)

    print(f"building token cache [{split_name}] rows={len(rows):,} ->", cache_path)
    start = time.perf_counter()
    tokenized_rows = []
    for idx, item in enumerate(rows, start=1):
        input_ids = tokenizer.encode(item["text"], add_bos_eos=True)
        input_ids = pad_or_truncate_token_ids(input_ids, SENTIMENT_EFFECTIVE_MAX_LENGTH, pad_id)
        tokenized_rows.append({
            "text": item["text"],
            "label": int(item["label"]),
            "input_ids": input_ids,
        })
        if idx % 20000 == 0:
            print(f"  tokenized {split_name}: {idx:,}/{len(rows):,}")

    torch.save({"metadata": cache_metadata, "data": tokenized_rows}, cache_path)
    elapsed = time.perf_counter() - start
    print(f"token cache saved [{split_name}]: {elapsed:.1f}s", cache_path)
    return tokenized_rows

DATA_DIR = Path(repo_dir) / "data"
train_jsonl = DATA_DIR / "nsmc_sentiment_train.jsonl"
val_jsonl = DATA_DIR / "nsmc_sentiment_val.jsonl"
test_jsonl = DATA_DIR / "nsmc_sentiment_test.jsonl"
if not train_jsonl.exists() or not val_jsonl.exists() or not test_jsonl.exists():
    raise FileNotFoundError("감성 분류 JSONL 파일이 없습니다. 먼저 2번 NSMC 데이터 준비 셀을 실행하세요.")

train_data = read_jsonl(train_jsonl)
val_data = read_jsonl(val_jsonl)
test_data = read_jsonl(test_jsonl)

if SENTIMENT_TRAIN_LIMIT is not None:
    train_data = train_data[:SENTIMENT_TRAIN_LIMIT]
if SENTIMENT_VAL_LIMIT is not None:
    val_data = val_data[:SENTIMENT_VAL_LIMIT]
if SENTIMENT_TEST_LIMIT is not None:
    test_data = test_data[:SENTIMENT_TEST_LIMIT]

token_cache_start = time.perf_counter()
train_data = load_or_build_tokenized_sentiment_split("train", train_data)
val_data = load_or_build_tokenized_sentiment_split("val", val_data)
test_data = load_or_build_tokenized_sentiment_split("test", test_data)
print("token cache prep time:", f"{time.perf_counter() - token_cache_start:.1f}s")

train_dataset = ReviewSentimentDataset(train_data, tokenizer, max_length=SENTIMENT_EFFECTIVE_MAX_LENGTH)
val_dataset = ReviewSentimentDataset(val_data, tokenizer, max_length=SENTIMENT_EFFECTIVE_MAX_LENGTH)
test_dataset = ReviewSentimentDataset(test_data, tokenizer, max_length=SENTIMENT_EFFECTIVE_MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=SENTIMENT_BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=SENTIMENT_BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=SENTIMENT_BATCH_SIZE, shuffle=False, drop_last=False)

print("data sizes:", {"train": len(train_dataset), "val": len(val_dataset), "test": len(test_dataset)})
print("batch counts:", {"train": len(train_loader), "val": len(val_loader), "test": len(test_loader)})

# -----------------------------------------------------------------------------
# 4. 학습/검증
# -----------------------------------------------------------------------------
optimizer_groups = []
lora_trainable_params = [
    param
    for name, param in sentiment_model.gpt.named_parameters()
    if param.requires_grad and (".lora_A." in name or ".lora_B." in name)
]
backbone_trainable_params = [
    param
    for name, param in sentiment_model.gpt.named_parameters()
    if param.requires_grad and not (".lora_A." in name or ".lora_B." in name)
]
classifier_trainable_params = [param for param in sentiment_model.classifier.parameters() if param.requires_grad]

if backbone_trainable_params:
    optimizer_groups.append({"params": backbone_trainable_params, "lr": SENTIMENT_BACKBONE_LR})
if lora_trainable_params:
    optimizer_groups.append({"params": lora_trainable_params, "lr": SENTIMENT_LORA_LR})
optimizer_groups.append({"params": classifier_trainable_params, "lr": SENTIMENT_CLASSIFIER_LR})

optimizer = torch.optim.AdamW(
    optimizer_groups,
    weight_decay=SENTIMENT_WEIGHT_DECAY,
)

history = []
best_val_loss = float("inf")
best_state = None
start_time = time.perf_counter()

for epoch in range(1, SENTIMENT_NUM_EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_loss, train_acc = train_epoch_sentiment(sentiment_model, train_loader, optimizer, device)
    val_loss, val_acc = evaluate_sentiment(sentiment_model, val_loader, device)
    epoch_time = time.perf_counter() - epoch_start

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "epoch_time_sec": round(epoch_time, 2),
    }
    history.append(row)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(sentiment_model.state_dict())

    print(
        f"epoch {epoch}: "
        f"train loss {train_loss:.4f}, train acc {train_acc:.4f}, "
        f"val loss {val_loss:.4f}, val acc {val_acc:.4f}, "
        f"time {epoch_time:.1f}s"
    )

if best_state is not None:
    sentiment_model.load_state_dict(best_state)

test_loss, test_acc = evaluate_sentiment(sentiment_model, test_loader, device)
total_time = time.perf_counter() - start_time

# -----------------------------------------------------------------------------
# 5. 결과 요약, confusion matrix, 오분류 예시
# -----------------------------------------------------------------------------
def prediction_report(model, dataset, max_examples: int = 8):
    loader = DataLoader(dataset, batch_size=SENTIMENT_BATCH_SIZE, shuffle=False)
    conf = torch.zeros(2, 2, dtype=torch.long)
    mistakes = []
    model.eval()
    offset = 0
    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device).long()
            logits = model(input_ids)
            preds = logits.argmax(dim=-1)
            for true_label, pred_label in zip(labels.cpu().tolist(), preds.cpu().tolist()):
                conf[true_label, pred_label] += 1
            for batch_idx, (true_label, pred_label) in enumerate(zip(labels.cpu().tolist(), preds.cpu().tolist())):
                if true_label != pred_label and len(mistakes) < max_examples:
                    item = dataset.data[offset + batch_idx]
                    mistakes.append({
                        "text": item["text"],
                        "true": true_label,
                        "pred": pred_label,
                    })
            offset += labels.size(0)
    return conf, mistakes

confusion, mistakes = prediction_report(sentiment_model, test_dataset)
label_name = {0: "부정", 1: "긍정"}

def pct(value: float) -> str:
    return f"{value * 100:.2f}%"

history_lines = ["| epoch | train loss | train acc | val loss | val acc | time sec |", "| ---: | ---: | ---: | ---: | ---: | ---: |"]
for row in history:
    history_lines.append(
        f"| {row['epoch']} | {row['train_loss']:.4f} | {pct(row['train_acc'])} | "
        f"{row['val_loss']:.4f} | {pct(row['val_acc'])} | {row['epoch_time_sec']:.2f} |"
    )

summary_md = f"""
### 감성 분류 결과 요약

| 항목 | 값 |
| --- | ---: |
| train samples | {len(train_dataset):,} |
| val samples | {len(val_dataset):,} |
| test samples | {len(test_dataset):,} |
| max_length | {SENTIMENT_EFFECTIVE_MAX_LENGTH} |
| batch_size | {SENTIMENT_BATCH_SIZE} |
| epochs | {SENTIMENT_NUM_EPOCHS} |
| freeze backbone | {SENTIMENT_FREEZE_BACKBONE} |
| use LoRA | {SENTIMENT_USE_LORA} |
| LoRA target modules | {", ".join(SENTIMENT_LORA_TARGET_MODULES)} |
| LoRA rank | {SENTIMENT_LORA_RANK} |
| LoRA alpha | {SENTIMENT_LORA_ALPHA:g} |
| unfreeze last blocks | {SENTIMENT_UNFREEZE_LAST_N_BLOCKS} |
| train GPT params | {trainable_counts["gpt_trainable"]:,} |
| train base GPT params | {trainable_counts["base_gpt_trainable"]:,} |
| train LoRA params | {trainable_counts["lora_trainable"]:,} |
| train classifier params | {trainable_counts["classifier_trainable"]:,} |
| token cache | {SENTIMENT_USE_TOKEN_CACHE} |
| backbone lr | {SENTIMENT_BACKBONE_LR:g} |
| LoRA lr | {SENTIMENT_LORA_LR:g} |
| classifier lr | {SENTIMENT_CLASSIFIER_LR:g} |
| best val loss | {best_val_loss:.4f} |
| test loss | {test_loss:.4f} |
| test accuracy | {pct(test_acc)} |
| total time min | {total_time / 60:.2f} |

### Epoch별 지표

{chr(10).join(history_lines)}

### Confusion Matrix - Test

| 실제 \\ 예측 | 부정(0) | 긍정(1) |
| --- | ---: | ---: |
| 부정(0) | {confusion[0, 0].item():,} | {confusion[0, 1].item():,} |
| 긍정(1) | {confusion[1, 0].item():,} | {confusion[1, 1].item():,} |
"""

if mistakes:
    summary_md += "\n### 오분류 예시\n\n| 실제 | 예측 | 리뷰 |\n| --- | --- | --- |\n"
    for item in mistakes:
        text = item["text"].replace("|", " ")[:120]
        summary_md += f"| {label_name[item['true']]} | {label_name[item['pred']]} | {text} |\n"

if display is not None and Markdown is not None:
    display(Markdown(summary_md))
else:
    print(summary_md)

# 저장: Colab Drive가 있으면 Drive runs/에, 없으면 local runs/에 저장합니다.
runs_root = Path("/content/drive/MyDrive/W14-gpt-lab/runs")
if not runs_root.exists():
    runs_root = Path("runs")
runs_root.mkdir(parents=True, exist_ok=True)
run_dir = runs_root / f"sentiment_{time.strftime('%Y%m%d_%H%M%S')}"
run_dir.mkdir(parents=True, exist_ok=False)

torch.save(
    {
        "model_state_dict": sentiment_model.state_dict(),
        "config": config,
        "history": history,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "confusion_matrix": confusion.tolist(),
    },
    run_dir / "sentiment_model.pt",
)
with (run_dir / "sentiment_summary.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "settings": {
                "max_length": SENTIMENT_EFFECTIVE_MAX_LENGTH,
                "batch_size": SENTIMENT_BATCH_SIZE,
                "num_epochs": SENTIMENT_NUM_EPOCHS,
                "freeze_backbone": SENTIMENT_FREEZE_BACKBONE,
                "unfreeze_last_n_blocks": SENTIMENT_UNFREEZE_LAST_N_BLOCKS,
                "train_final_layernorm": SENTIMENT_TRAIN_FINAL_LAYERNORM,
                "use_lora": SENTIMENT_USE_LORA,
                "lora_info": lora_info,
                "trainable_counts": trainable_counts,
                "use_token_cache": SENTIMENT_USE_TOKEN_CACHE,
                "backbone_lr": SENTIMENT_BACKBONE_LR,
                "lora_lr": SENTIMENT_LORA_LR,
                "classifier_lr": SENTIMENT_CLASSIFIER_LR,
                "weight_decay": SENTIMENT_WEIGHT_DECAY,
                "train_limit": SENTIMENT_TRAIN_LIMIT,
                "val_limit": SENTIMENT_VAL_LIMIT,
                "test_limit": SENTIMENT_TEST_LIMIT,
            },
            "history": history,
            "best_val_loss": best_val_loss,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "confusion_matrix": confusion.tolist(),
            "mistakes": mistakes,
            "total_time_sec": round(total_time, 2),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )
print("saved sentiment run:", run_dir)
